In [10]:
import pandas as pd
import numpy as np

In [11]:
def select_best_ensemble_k(model_files, k=3, keep_top_n=7):
    """
    model_files: 字典 {模型名稱: 檔案路徑}
    k: 最終要選出來投票的模型數量
    keep_top_n: 初始篩選時要保留的前幾個模型 (依正確率排序)
    """
    
    results_wrong_vec = {} # 儲存每個模型是否答錯的向量 (1為錯, 0為對)
    accuracies = {}
    
    # 1. 讀取與計算基本數據
    for name, path in model_files.items():
        try:
            df = pd.read_excel(path)
            # 確保資料格式一致
            actual = df['Actual'].astype(str).str.strip()
            predicted = df['Predicted'].fillna(0).astype(int).astype(str).str.strip()
            
            # 計算正確率
            is_correct = (actual == predicted)
            accuracies[name] = is_correct.mean()
            
            # 記錄答錯位置 (True=1, False=0)
            results_wrong_vec[name] = (~is_correct).astype(int).values
        except Exception as e:
            print(f"讀取 {name} 失敗: {e}")
            continue
    
    # 2. 依照正確率排序並篩選
    sorted_by_acc = sorted(accuracies.items(), key=lambda x: x[1], reverse=True)
    top_models = [m[0] for m in sorted_by_acc[:keep_top_n]]
    
    print(f"--- 模型篩選結果 (保留前 {keep_top_n} 名) ---")
    for name in top_models:
        print(f"{name}: {accuracies[name]:.2%}")

    # 3. 計算雙重錯誤率矩陣
    num_questions = len(next(iter(results_wrong_vec.values())))
    de_matrix = pd.DataFrame(index=top_models, columns=top_models)

    # 用於尋找初始最小 Double Error 對
    min_de_val = 1.0
    initial_pair = []

    for m1 in top_models:
        for m2 in top_models:
            if m1 == m2:
                de_matrix.loc[m1, m2] = np.nan
            else:
                # 兩者同時答錯率
                both_wrong = np.sum((results_wrong_vec[m1] == 1) & (results_wrong_vec[m2] == 1))
                de_val = both_wrong / num_questions
                de_matrix.loc[m1, m2] = de_val
                
                # 紀錄全域最小的 Double Error 對
                if de_val < min_de_val:
                    min_de_val = de_val
                    initial_pair = [m1, m2]

    # 儲存矩陣
    de_matrix.to_csv("double_error_matrix.csv")

    # 4. 貪婪演算法選出 K 個模型
    # 【修正】：起點改為 Double Error 最小的兩個模型
    selected = initial_pair
    remaining = [m for m in top_models if m not in selected]

    print(f"\n--- 迭代選擇過程 ---")
    print(f"Round 1 & 2: 初始互補模型對 (Double Error 最小) = {selected}, DE = {min_de_val:.6f}")

    while len(selected) < k and remaining:
        scores = {}
        print(f"\nRound {len(selected) + 1} 評分中...")
        for candidate in remaining:
            # 計算該候選模型與已選模型群組的平均 Double Error
            de_list = [de_matrix.loc[candidate, s] for s in selected]
            avg_de = np.mean(de_list)
            scores[candidate] = avg_de
            print(f"  > 候選 {candidate}: 與 {selected} 的錯誤率分別為 {de_list}, 平均 = {avg_de:.6f}")
        
        # 選擇平均 Double Error 最小的模型
        best_candidate = min(scores, key=scores.get)
        print(f"決策：選擇平均錯誤率最低的 [{best_candidate}]")
        
        selected.append(best_candidate)
        remaining.remove(best_candidate)
        
    return selected, de_matrix

In [12]:
MODELS = ["gemma3_12b", "qwen3_8b", "deepseek-r1_8b", "llama3.1_8b", "mistral_7b", 
          "huihui_ai_internlm3-abliterated", "ingu627_phi-4-mini-instruct-q4", 
          "glm4_9b", "myaniu_minicpm4-8b_Q4_K_M"]

# 修正檔名格式以符合 result_*.xlsx 的命名規則
MODEL_FILE_MAP = {m: f"result_{m.replace(':', '_').replace('/', '_')}.xlsx" for m in MODELS}

# 執行選擇邏輯
final_models, full_matrix = select_best_ensemble_k(MODEL_FILE_MAP, k=3, keep_top_n=9)
print(f"\n最終建議投票的組合: {final_models}")

--- 模型篩選結果 (保留前 9 名) ---
huihui_ai_internlm3-abliterated: 94.11%
glm4_9b: 93.46%
qwen3_8b: 93.36%
gemma3_12b: 92.86%
myaniu_minicpm4-8b_Q4_K_M: 91.31%
deepseek-r1_8b: 90.41%
llama3.1_8b: 89.61%
ingu627_phi-4-mini-instruct-q4: 83.52%
mistral_7b: 81.47%

--- 迭代選擇過程 ---
Round 1 & 2: 初始互補模型對 (Double Error 最小) = ['huihui_ai_internlm3-abliterated', 'qwen3_8b'], DE = 0.032967

Round 3 評分中...
  > 候選 glm4_9b: 與 ['huihui_ai_internlm3-abliterated', 'qwen3_8b'] 的錯誤率分別為 [np.float64(0.03396603396603397), np.float64(0.03796203796203796)], 平均 = 0.035964
  > 候選 gemma3_12b: 與 ['huihui_ai_internlm3-abliterated', 'qwen3_8b'] 的錯誤率分別為 [np.float64(0.036463536463536464), np.float64(0.03896103896103896)], 平均 = 0.037712
  > 候選 myaniu_minicpm4-8b_Q4_K_M: 與 ['huihui_ai_internlm3-abliterated', 'qwen3_8b'] 的錯誤率分別為 [np.float64(0.03596403596403597), np.float64(0.040459540459540456)], 平均 = 0.038212
  > 候選 deepseek-r1_8b: 與 ['huihui_ai_internlm3-abliterated', 'qwen3_8b'] 的錯誤率分別為 [np.float64(0.03796203796203796), np.flo

In [13]:
full_matrix

,huihui_ai_internlm3-abliterated,glm4_9b,qwen3_8b,gemma3_12b,myaniu_minicpm4-8b_Q4_K_M,deepseek-r1_8b,llama3.1_8b,ingu627_phi-4-mini-instruct-q4,mistral_7b
huihui_ai_internlm3-abliterated,NaN,0.033966,0.032967,0.036464,0.035964,0.037962,0.041459,0.04046,0.041459
glm4_9b,0.033966,NaN,0.037962,0.03996,0.040959,0.040959,0.045455,0.046454,0.045954
qwen3_8b,0.032967,0.037962,NaN,0.038961,0.04046,0.053447,0.039461,0.046454,0.048951
gemma3_12b,0.036464,0.03996,0.038961,NaN,0.03996,0.040959,0.047453,0.051948,0.04995
myaniu_minicpm4-8b_Q4_K_M,0.035964,0.040959,0.04046,0.03996,NaN,0.047453,0.048951,0.056444,0.058941
deepseek-r1_8b,0.037962,0.040959,0.053447,0.040959,0.047453,NaN,0.04995,0.059441,0.064436
llama3.1_8b,0.041459,0.045455,0.039461,0.047453,0.048951,0.04995,NaN,0.068432,0.072428
ingu627_phi-4-mini-instruct-q4,0.04046,0.046454,0.046454,0.051948,0.056444,0.059441,0.068432,NaN,0.097902
mistral_7b,0.041459,0.045954,0.048951,0.04995,0.058941,0.064436,0.072428,0.097902,NaN
